# 01 Data Preparation: Master Panel

This notebook constructs the realised-weather store-category-day panel used by the basket, regression, synthetic-weather, and machine-learning notebooks. It preserves the category, campaign, calendar, closure, and realised-weather merge rules from the legacy workflow while writing standardized project-root-relative outputs.


## Pipeline position

This notebook follows `notebooks/00_setup_and_data_dictionary.ipynb` and precedes `notebooks/02_basket_analysis.ipynb`. It is the first active notebook that reads the full transaction file, so it should only be run when the prepared realised-weather panels need to be rebuilt.


## Inputs

The notebook reads `data/Dataset.parquet`, `data/weather_final_master_all_vars.parquet`, and `data/processed/realised_weather_daily_windows.parquet`. The legacy `data/MASTERFILE_CALANDER.parquet` file is used only as a reference checkpoint and is not overwritten.


## Outputs

When executed, the notebook writes the standardized realised-weather panels and traceability checks below. The legacy `data/MASTERFILE_CALANDER.parquet` file is not overwritten.

- `data/interim/df_final_analysis_ready.parquet`
- `data/interim/df_catday_full_calendar.parquet`
- `data/processed/master_panel_realized_weather.parquet`
- `data/processed/master_panel_realized_weather_windows.parquet`
- `data/processed/regression_panel_realized_weather.parquet`
- `data/processed/weather_observed_daily.parquet`
- `data/processed/descriptive_sales_weather_panel.parquet`
- `outputs/data_preparation/master_panel_checks.csv`
- `outputs/data_preparation/master_panel_realized_weather_windows_checks.csv`

`master_panel_realized_weather.parquet` is the clean split-pipeline replacement for the legacy master panel. `descriptive_sales_weather_panel.parquet` is reporting-only, while `master_panel_realized_weather_windows.parquet` adds a long-format weather-window version for statistical window comparison.


## Methodological notes

This notebook creates realised historical panels only. Realised weather is valid for regression, synthetic-weather calibration, diagnostics, and observed-weather history construction, but realised target-day weather must not be used later as operational ML forecast-weather input. Forecasting notation remains `t` for the origin and `t+h` for the target date. Later sales lags and rolling features must use only information available at origin `t`. Regression uses `log(1 + Q_itk)` for store-category-day sales, while the main ML target remains original-scale `Q_itk` / `Antall_total`. The weather-window panel supports comparison of `full_day_00_23`, `trade_08_18`, and `trade_08_19`.


## Environment and portable paths

The notebook is intended for the `csvi_env` kernel. Paths are discovered from the project root and are not hardcoded to a local machine path.


In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd

MARKER_FILES = ("README_FOR_CODEX.md", "AGENTS.md", "pyproject.toml")


def find_project_root(start=None):
    """Walk upward until a project marker file is found."""
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if any((candidate / marker).exists() for marker in MARKER_FILES):
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from inside the project "
        "folder or make sure README_FOR_CODEX.md, AGENTS.md, or pyproject.toml exists."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DATA_PREP_DIR = OUTPUT_DIR / "data_preparation"

DATASET_PATH = DATA_DIR / "Dataset.parquet"
WEATHER_OBS_PATH = DATA_DIR / "weather_final_master_all_vars.parquet"
REALISED_WEATHER_WINDOWS_PATH = PROCESSED_DIR / "realised_weather_daily_windows.parquet"
LEGACY_MASTER_PATH = DATA_DIR / "MASTERFILE_CALANDER.parquet"

ANALYSIS_READY_PATH = INTERIM_DIR / "df_final_analysis_ready.parquet"
FULL_CALENDAR_PATH = INTERIM_DIR / "df_catday_full_calendar.parquet"
MASTER_PANEL_PATH = PROCESSED_DIR / "master_panel_realized_weather.parquet"
MASTER_PANEL_WINDOWS_PATH = PROCESSED_DIR / "master_panel_realized_weather_windows.parquet"
REGRESSION_PANEL_PATH = PROCESSED_DIR / "regression_panel_realized_weather.parquet"
WEATHER_OBSERVED_DAILY_PATH = PROCESSED_DIR / "weather_observed_daily.parquet"
DESCRIPTIVE_PANEL_PATH = PROCESSED_DIR / "descriptive_sales_weather_panel.parquet"
CHECKS_PATH = OUTPUT_DATA_PREP_DIR / "master_panel_checks.csv"
WINDOW_PANEL_CHECKS_PATH = OUTPUT_DATA_PREP_DIR / "master_panel_realized_weather_windows_checks.csv"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DATA_PREP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


def project_relative(path):
    return path.relative_to(PROJECT_ROOT).as_posix()


def require_file(path, description):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {project_relative(path)}. "
            "Place the file under the documented project folder before running this notebook."
        )


def require_columns(frame, required_columns, frame_name):
    missing = [column for column in required_columns if column not in frame.columns]
    if missing:
        raise KeyError(f"{frame_name} is missing required columns: {missing}")


print(f"Project root: {PROJECT_ROOT}")
print(f"Master panel output: {project_relative(MASTER_PANEL_PATH)}")


## Input availability checks

The next cell verifies file existence and lightweight metadata without loading the full datasets.


In [ ]:
input_files = [
    (DATASET_PATH, "transaction source data"),
    (WEATHER_OBS_PATH, "legacy observed weather source data"),
    (REALISED_WEATHER_WINDOWS_PATH, "realised weather window source from notebook 01a"),
]

for path, description in input_files:
    require_file(path, description)

input_summary = []
for path, description in input_files + [(LEGACY_MASTER_PATH, "legacy realised-weather master panel")]:
    if path.exists():
        input_summary.append(
            {
                "file": project_relative(path),
                "description": description,
                "exists": True,
                "size_mb": round(path.stat().st_size / 1_000_000, 2),
            }
        )
    else:
        input_summary.append(
            {"file": project_relative(path), "description": description, "exists": False, "size_mb": np.nan}
        )

input_summary_df = pd.DataFrame(input_summary)
display(input_summary_df)

## Memory-first transaction processing

The transaction source contains approximately 16-20 million rows. The implementation reads only required columns, optimizes dtypes early, processes the Parquet file in chunks, deletes large intermediates, and reports approximate memory use after major transformations. Filtering, category definitions, campaign rules, and aggregation definitions follow the legacy workflow.


In [ ]:
REQUIRED_TRANSACTION_COLUMNS = [
    "DatoSolgt",
    "AvdelingID",
    "AvdelingTekst",
    "ArtikkelNavn",
    "HovedartikkelgruppeTxt",
    "Artikkelkategori1Txt",
    "Produktgruppe",
    "OrdrekildeTekst",
    "PrisgruppeTekst",
    "RabattKr",
    "NettoKr",
    "AntallArtikler",
    "OrdreID",
    "Kommune",
    "Region",
    "Årstid",
    "Helligdager",
]

TEXT_COLUMNS_TO_CATEGORY = [
    "AvdelingTekst",
    "HovedartikkelgruppeTxt",
    "Artikkelkategori1Txt",
    "Produktgruppe",
    "OrdrekildeTekst",
    "PrisgruppeTekst",
    "Kommune",
    "Region",
    "Årstid",
]


def report_memory(frame, name):
    """Print shape and approximate dataframe memory use in GB."""
    memory_gb = frame.memory_usage(deep=True).sum() / 1_000_000_000
    print(f"{name}: shape={frame.shape}, approx_memory={memory_gb:.3f} GB")
    return memory_gb


def downcast_numeric_columns(frame):
    """Downcast only columns where the current pipeline does not require high precision."""
    if "AvdelingID" in frame.columns:
        frame["AvdelingID"] = pd.to_numeric(frame["AvdelingID"], errors="raise", downcast="integer")
    if "RabattKr" in frame.columns:
        frame["RabattKr"] = pd.to_numeric(frame["RabattKr"], errors="coerce", downcast="float")
    if "AntallArtikler" in frame.columns:
        frame["AntallArtikler"] = pd.to_numeric(frame["AntallArtikler"], errors="coerce", downcast="float")
    if "NettoKr" in frame.columns:
        frame["NettoKr"] = pd.to_numeric(frame["NettoKr"], errors="coerce")
    return frame


def optimize_string_columns(frame, columns, max_category_share=0.50):
    """Convert repeated text columns to category dtype when this is likely memory-saving."""
    n_rows = len(frame)
    if n_rows == 0:
        return frame
    for column in columns:
        if column in frame.columns and frame[column].dtype == "object":
            unique_share = frame[column].nunique(dropna=False) / n_rows
            if unique_share <= max_category_share:
                frame[column] = frame[column].astype("category")
    return frame


def bool_array(mask):
    """Return a dense bool numpy array, treating missing values as False."""
    if isinstance(mask, pd.Series):
        return mask.fillna(False).to_numpy(dtype=bool)
    return np.asarray(mask, dtype=bool)


def parquet_column_status(path, required_columns):
    """Inspect Parquet schema metadata without loading the full dataset."""
    try:
        import pyarrow.parquet as pq

        available_columns = set(pq.ParquetFile(path).schema_arrow.names)
        return pd.DataFrame(
            {
                "column": required_columns,
                "available": [column in available_columns for column in required_columns],
            }
        )
    except Exception as exc:
        return pd.DataFrame(
            {"column": required_columns, "available": pd.NA, "note": f"Schema inspection failed: {exc}"}
        )


transaction_column_status = parquet_column_status(DATASET_PATH, REQUIRED_TRANSACTION_COLUMNS)
display(transaction_column_status)

## Product filtering and analysis categories

The following functions preserve the exclusion and `Analyse_Kategori` rules from the legacy notebook while applying them batch by batch. Copies are limited to filtered chunks rather than the full transaction dataset.


In [ ]:
excluded_main_groups = ["Non-food"]
excluded_categories = ["Kioskvarer", "Catering", "Tilbehør", "Alkoholdrikk"]
excluded_product_groups = ["Vin", "Øl", "Tilbehør"]
analysis_categories = ["Dinner", "Lunch", "Sweets", "Salads", "Dessert", "Cold Drinks", "Hot Drinks"]
analysis_category_dtype = pd.CategoricalDtype(categories=[*analysis_categories, "Ekskluderes"])

specific_campaign_products = [
    "Club sandwich 149,-",
    "Cæsar tilbud",
    "Club sandwich 199,-",
    "Kampanje",
    "Rødbeteburger - tilbud",
    "Cæsar tilbud",
    "Classic Burger tilbud",
    "Kremet pasta tilbud",
    "Nachos tilbud",
    "Osteblings tilbud",
    "Rekesalat tilbud",
    "Chili sin Carne - tilbud",
    "Barnebrett tilbud",
    "Club Sandwich Tilbud",
    "Drømmevaffel Tilbud",
    "Grønnsaksuppe tilbud",
    "Sesongburger tilbud",
    "Strømmen kupongtilbud",
    "Suppe tilbud",
    "Tomat & mozzarellasalat Tilbud",
    "Lasagne tilbud",
]
campaign_name_pattern = "|".join(specific_campaign_products)

daily_index_columns = [
    "DatoSolgt",
    "AvdelingID",
    "AvdelingTekst",
    "ArtikkelNavn",
    "HovedartikkelgruppeTxt",
    "Artikkelkategori1Txt",
    "Produktgruppe",
    "Analyse_Kategori",
    "Kommune",
    "Region",
    "Ukedag",
    "Årstid",
    "Helligdager",
    "Is_Campaign",
]

group_columns = [
    "DatoSolgt",
    "Ukedag",
    "Årstid",
    "Helligdager",
    "AvdelingID",
    "AvdelingTekst",
    "Kommune",
    "Region",
    "Analyse_Kategori",
    "Is_Campaign",
]


def safe_filter_transactions(transactions_chunk):
    """Apply the exact Hovedfilen row filters to one chunk without copying the full dataset."""
    require_columns(transactions_chunk, REQUIRED_TRANSACTION_COLUMNS, "transactions_chunk")
    transactions_chunk = downcast_numeric_columns(transactions_chunk)
    transactions_chunk = optimize_string_columns(transactions_chunk, TEXT_COLUMNS_TO_CATEGORY)

    keep_mask = np.ones(len(transactions_chunk), dtype=bool)
    keep_mask &= ~bool_array(transactions_chunk["HovedartikkelgruppeTxt"].isin(excluded_main_groups))
    keep_mask &= ~bool_array(transactions_chunk["Artikkelkategori1Txt"].isin(excluded_categories))
    keep_mask &= ~bool_array(transactions_chunk["Produktgruppe"].isin(excluded_product_groups))
    keep_mask &= bool_array(transactions_chunk["OrdrekildeTekst"] != "Netthandel")
    keep_mask &= bool_array(transactions_chunk["ArtikkelNavn"] != "Diverse brød")
    keep_mask &= bool_array(transactions_chunk["Artikkelkategori1Txt"] != "Småretter")

    # Chunk-sized shallow copy before adding derived columns; avoids a full raw-data copy.
    filtered = transactions_chunk.loc[keep_mask].copy(deep=False)
    del keep_mask

    filtered["DatoSolgt"] = pd.to_datetime(filtered["DatoSolgt"])
    article_name = filtered["ArtikkelNavn"].astype("string")
    category_1 = filtered["Artikkelkategori1Txt"]

    category_values = np.full(len(filtered), "Ekskluderes", dtype=object)

    mask = bool_array(
        article_name.str.contains("Club sandwich", case=False, na=False)
        | category_1.isin(["Burger", "Middag"])
    )
    category_values[mask] = "Dinner"
    del mask

    mask = bool_array(article_name.str.contains("Focaccia", case=False, na=False) | (category_1 == "Brødmat"))
    category_values[mask] = "Lunch"
    del mask

    mask = bool_array(category_1.isin(["Bakst", "Kaker"]))
    category_values[mask] = "Sweets"
    del mask

    mask = bool_array(category_1 == "Salat")
    category_values[mask] = "Salads"
    del mask

    mask = bool_array(category_1 == "Dessert")
    category_values[mask] = "Dessert"
    del mask

    mask = bool_array(category_1 == "Kald drikke")
    category_values[mask] = "Cold Drinks"
    del mask

    mask = bool_array(category_1 == "Varm drikke")
    category_values[mask] = "Hot Drinks"
    del mask, category_1

    filtered["Analyse_Kategori"] = pd.Series(
        category_values,
        index=filtered.index,
        dtype=analysis_category_dtype,
    )
    del category_values

    campaign_mask = (
        bool_array(filtered["PrisgruppeTekst"] != "Ordinær")
        | bool_array(filtered["RabattKr"] > 0)
        | bool_array(article_name.str.contains(campaign_name_pattern, case=False, na=False))
    )
    filtered["Is_Campaign"] = campaign_mask.astype("int8")
    filtered["AntallSalg"] = filtered["OrdreID"].notna().astype("int32")
    filtered = filtered.drop(columns=["OrdreID"])
    filtered["Ukedag"] = filtered["DatoSolgt"].dt.day_name().astype("category")

    del article_name, campaign_mask
    return filtered


def combine_daily_parts(daily_parts):
    """Combine already-aggregated chunk outputs and re-aggregate duplicate keys."""
    if not daily_parts:
        return pd.DataFrame(columns=[*daily_index_columns, "NettoKr", "AntallArtikler", "AntallSalg"])
    combined = pd.concat(daily_parts, ignore_index=True, copy=False)
    combined = (
        combined.groupby(daily_index_columns, observed=True, as_index=False)
        .agg({"NettoKr": "sum", "AntallArtikler": "sum", "AntallSalg": "sum"})
    )
    combined["Is_Campaign"] = combined["Is_Campaign"].fillna(0).astype("int8")
    return combined


def process_transaction_chunk(raw_chunk, chunk_name):
    """Filter, categorize, campaign-mark, and aggregate one transaction chunk."""
    raw_rows = len(raw_chunk)
    filtered = safe_filter_transactions(raw_chunk)
    filtered_rows = len(filtered)

    category_counts = filtered["Analyse_Kategori"].value_counts(dropna=False)
    date_min = filtered["DatoSolgt"].min() if filtered_rows else pd.NaT
    date_max = filtered["DatoSolgt"].max() if filtered_rows else pd.NaT
    stores = set(filtered["AvdelingID"].dropna().unique()) if filtered_rows else set()

    df_daily_chunk = (
        filtered.groupby(daily_index_columns, observed=True, as_index=False)
        .agg({"NettoKr": "sum", "AntallArtikler": "sum", "AntallSalg": "sum"})
    )

    stats = {
        "chunk": chunk_name,
        "raw_rows": raw_rows,
        "filtered_rows": filtered_rows,
        "date_min": date_min,
        "date_max": date_max,
        "stores": stores,
    }

    del filtered
    gc.collect()
    return df_daily_chunk, stats, category_counts

## Campaign variables and daily aggregation

This step builds the transaction-level campaign indicators and aggregates sales to the store-category-day level. It preserves the legacy campaign-product list and aggregation logic, then writes the aggregated output under `data/interim/`.


In [ ]:
def build_grouped_sales_from_transactions(path, batch_size=500_000, combine_every=8):
    """Build df_final_grouped from the transaction Parquet using chunked processing when possible."""
    processing_stats = {
        "read_strategy": "pyarrow.dataset chunked",
        "raw_rows": 0,
        "filtered_rows": 0,
        "date_min": pd.NaT,
        "date_max": pd.NaT,
        "stores": set(),
    }
    category_counts_total = pd.Series(dtype="int64")
    daily_parts = []

    try:
        import pyarrow.dataset as ds

        dataset = ds.dataset(path, format="parquet")
        available_columns = set(dataset.schema.names)
        missing_columns = [
            column for column in REQUIRED_TRANSACTION_COLUMNS if column not in available_columns
        ]
        if missing_columns:
            raise KeyError(f"Transaction Parquet is missing required columns: {missing_columns}")

        scanner = dataset.scanner(columns=REQUIRED_TRANSACTION_COLUMNS, batch_size=batch_size)
        for chunk_number, record_batch in enumerate(scanner.to_batches(), start=1):
            raw_chunk = record_batch.to_pandas()
            df_daily_chunk, chunk_stats, category_counts = process_transaction_chunk(
                raw_chunk,
                chunk_name=f"batch_{chunk_number}",
            )

            processing_stats["raw_rows"] += chunk_stats["raw_rows"]
            processing_stats["filtered_rows"] += chunk_stats["filtered_rows"]
            processing_stats["stores"].update(chunk_stats["stores"])
            if pd.notna(chunk_stats["date_min"]):
                processing_stats["date_min"] = min(
                    processing_stats["date_min"], chunk_stats["date_min"]
                ) if pd.notna(processing_stats["date_min"]) else chunk_stats["date_min"]
                processing_stats["date_max"] = max(
                    processing_stats["date_max"], chunk_stats["date_max"]
                ) if pd.notna(processing_stats["date_max"]) else chunk_stats["date_max"]

            category_counts_total = category_counts_total.add(
                category_counts,
                fill_value=0,
            ).astype("int64")
            if not df_daily_chunk.empty:
                daily_parts.append(df_daily_chunk)

            if len(daily_parts) >= combine_every:
                daily_parts = [combine_daily_parts(daily_parts)]
                report_memory(daily_parts[0], f"combined daily parts after batch {chunk_number}")

            del raw_chunk, df_daily_chunk, record_batch
            gc.collect()

            if chunk_number % 10 == 0:
                print(
                    f"Processed {chunk_number:,} batches; "
                    f"raw rows={processing_stats['raw_rows']:,}; "
                    f"kept rows={processing_stats['filtered_rows']:,}"
                )

    except ImportError:
        processing_stats["read_strategy"] = "pandas selected-column fallback"
        print("pyarrow.dataset is unavailable; using pandas selected-column fallback.")
        raw_chunk = pd.read_parquet(path, columns=REQUIRED_TRANSACTION_COLUMNS)
        report_memory(raw_chunk, "selected transaction columns before filtering")
        df_daily_chunk, chunk_stats, category_counts_total = process_transaction_chunk(
            raw_chunk,
            "pandas_full_selected",
        )
        daily_parts.append(df_daily_chunk)
        processing_stats.update(
            raw_rows=chunk_stats["raw_rows"],
            filtered_rows=chunk_stats["filtered_rows"],
            date_min=chunk_stats["date_min"],
            date_max=chunk_stats["date_max"],
            stores=chunk_stats["stores"],
        )
        del raw_chunk, df_daily_chunk
        gc.collect()

    df_daily_combined = combine_daily_parts(daily_parts)
    daily_rows = len(df_daily_combined)
    report_memory(df_daily_combined, "df_daily combined")

    df_final = (
        df_daily_combined.groupby(group_columns, observed=True, as_index=False)
        .agg({"NettoKr": "sum", "AntallArtikler": "sum", "AntallSalg": "sum"})
    )
    report_memory(df_final, "df_final_grouped")

    del daily_parts, df_daily_combined
    gc.collect()

    processing_stats["n_stores"] = len(processing_stats.pop("stores"))
    processing_stats["df_daily_rows"] = daily_rows
    processing_stats["df_final_grouped_rows"] = len(df_final)

    category_counts_df = (
        category_counts_total.sort_values(ascending=False)
        .rename_axis("Analyse_Kategori")
        .reset_index(name="rows")
    )

    return df_final, processing_stats, category_counts_df


df_final_grouped, transaction_processing_stats, category_counts = (
    build_grouped_sales_from_transactions(DATASET_PATH)
)

raw_transaction_rows = transaction_processing_stats["raw_rows"]
filtered_transaction_rows = transaction_processing_stats["filtered_rows"]
df_daily_rows = transaction_processing_stats["df_daily_rows"]
df_final_grouped_rows = transaction_processing_stats["df_final_grouped_rows"]

df_final_grouped.to_parquet(ANALYSIS_READY_PATH, index=False)

print(f"Read strategy: {transaction_processing_stats['read_strategy']}")
print(f"Raw transaction rows processed: {raw_transaction_rows:,}")
print(f"Rows after filtering: {filtered_transaction_rows:,}")
print(
    f"Date range: {transaction_processing_stats['date_min'].date()} "
    f"to {transaction_processing_stats['date_max'].date()}"
)
print(f"Stores: {transaction_processing_stats['n_stores']:,}")
print(f"Grouped daily/category rows: {len(df_final_grouped):,}")
print(f"Saved: {project_relative(ANALYSIS_READY_PATH)}")
display(category_counts)

## Store-category-day sales panel

The following cell creates the sales and campaign variables used in the realised-weather master panel, including the preserved `log_Salg = log(1 + Antall_total)` definition.


In [ ]:
df_final_grouped_rows = len(df_final_grouped)
# Copy only the grouped table, not the raw transaction table.
sales_panel_source = df_final_grouped.loc[df_final_grouped["AntallArtikler"] >= 0].copy()
sales_panel_source.replace([np.inf, -np.inf], np.nan, inplace=True)
report_memory(sales_panel_source, "sales_panel_source after non-negative filter")

sales_panel_source["Antall_total"] = sales_panel_source["AntallArtikler"]
sales_panel_source["Antall_campaign"] = np.where(
    sales_panel_source["Is_Campaign"] == 1,
    sales_panel_source["AntallArtikler"],
    0,
)
sales_panel_source["Antall_ordinary"] = np.where(
    sales_panel_source["Is_Campaign"] == 0,
    sales_panel_source["AntallArtikler"],
    0,
)

catday_group_columns = [
    "DatoSolgt",
    "AvdelingID",
    "AvdelingTekst",
    "Region",
    "Analyse_Kategori",
    "Ukedag",
    "Årstid",
    "Helligdager",
]

df_catday = (
    sales_panel_source.groupby(catday_group_columns, observed=True, as_index=False)
    .agg(
        Antall_total=("Antall_total", "sum"),
        Antall_campaign=("Antall_campaign", "sum"),
        Antall_ordinary=("Antall_ordinary", "sum"),
    )
)

df_catday["CampaignShare"] = np.where(
    df_catday["Antall_total"] > 0,
    df_catday["Antall_campaign"] / df_catday["Antall_total"],
    0,
)
df_catday["CampaignDay"] = (df_catday["Antall_campaign"] > 0).astype("int8")
df_catday["log_Salg"] = np.log1p(df_catday["Antall_total"])

report_memory(df_catday, "df_catday")
del sales_panel_source
gc.collect()

print(f"Store-category-day rows before calendar expansion: {len(df_catday):,}")
print(f"Categories: {df_catday['Analyse_Kategori'].nunique():,}")

## Full-calendar panel

This step expands observed sales to a complete date-store-category grid from 2022-01-01 to 2025-07-31. Missing store-category-day sales are filled with zero to preserve the full calendar structure.


In [ ]:
START_DATE = "2022-01-01"
END_DATE = "2025-07-31"

df_catday["DatoSolgt"] = pd.to_datetime(df_catday["DatoSolgt"])
key_columns = ["DatoSolgt", "AvdelingID", "Analyse_Kategori"]
duplicate_keys = df_catday.duplicated(key_columns).sum()
assert duplicate_keys == 0, f"df_catday has duplicate keys on {key_columns}: {duplicate_keys}"

optional_store_columns = [
    column for column in ["AvdelingTekst", "Kommune", "Region"] if column in df_catday.columns
]
store_map = df_catday[["AvdelingID", *optional_store_columns]].drop_duplicates()

for column in ["AvdelingTekst", "Region"]:
    if column not in store_map.columns:
        raise KeyError(f"Missing store metadata column required by current logic: {column}")
    inconsistent = store_map.groupby("AvdelingID")[column].nunique(dropna=False)
    if (inconsistent > 1).any():
        bad_stores = inconsistent[inconsistent > 1].index.tolist()[:10]
        raise ValueError(f"Store metadata column {column} has multiple values for stores: {bad_stores}")

store_map = store_map.drop_duplicates("AvdelingID")


def season_from_month(month):
    if month in [12, 1, 2]:
        return "Vinter"
    if month in [3, 4, 5]:
        return "Vår"
    if month in [6, 7, 8]:
        return "Sommer"
    return "Høst"


dates = pd.date_range(START_DATE, END_DATE, freq="D")
date_dim = pd.DataFrame({"DatoSolgt": dates})
date_dim["Ukedag"] = date_dim["DatoSolgt"].dt.day_name().astype("category")
date_dim["Årstid"] = date_dim["DatoSolgt"].dt.month.map(season_from_month).astype("category")

try:
    import holidays

    no_holidays = holidays.country_holidays("NO", years=range(2022, 2026))
    date_dim["HelligdagNavn"] = date_dim["DatoSolgt"].dt.date.map(
        lambda date_value: no_holidays.get(date_value, "")
    )
    date_dim["Helligdager"] = (date_dim["HelligdagNavn"] != "").astype("int8")
    date_dim["HelligdagNavn"] = date_dim["HelligdagNavn"].replace("", "Ingen")
except Exception as exc:
    print(f"Holiday package unavailable or failed ({exc}); using non-holiday fallback.")
    date_dim["HelligdagNavn"] = "Ingen"
    date_dim["Helligdager"] = np.int8(0)

stores = pd.Index(df_catday["AvdelingID"].dropna().unique(), name="AvdelingID")
categories = pd.Index(df_catday["Analyse_Kategori"].dropna().unique(), name="Analyse_Kategori")

full_index = pd.MultiIndex.from_product(
    [dates, stores, categories],
    names=key_columns,
).to_frame(index=False)
full_calendar_panel = full_index.merge(
    store_map,
    on="AvdelingID",
    how="left",
).merge(date_dim, on="DatoSolgt", how="left")

measure_columns = [
    "Antall_total",
    "Antall_campaign",
    "Antall_ordinary",
    "CampaignShare",
    "CampaignDay",
    "log_Salg",
]
full_calendar_panel = full_calendar_panel.merge(
    df_catday[key_columns + measure_columns],
    on=key_columns,
    how="left",
)

for column in ["Antall_total", "Antall_campaign", "Antall_ordinary"]:
    full_calendar_panel[column] = full_calendar_panel[column].fillna(0)

full_calendar_panel["CampaignShare"] = np.where(
    full_calendar_panel["Antall_total"] > 0,
    full_calendar_panel["Antall_campaign"] / full_calendar_panel["Antall_total"],
    0,
)
full_calendar_panel["CampaignDay"] = (full_calendar_panel["Antall_campaign"] > 0).astype("int8")
full_calendar_panel["log_Salg"] = np.log1p(full_calendar_panel["Antall_total"])

for column in ["Ukedag_old", "Årstid_old", "Helligdager_old"]:
    if column in full_calendar_panel.columns:
        full_calendar_panel = full_calendar_panel.drop(columns=column)

expected_rows = len(dates) * len(stores) * len(categories)
assert len(full_calendar_panel) == expected_rows, (len(full_calendar_panel), expected_rows)
assert full_calendar_panel.duplicated(key_columns).sum() == 0

front_columns = [
    "DatoSolgt",
    "AvdelingID",
    "AvdelingTekst",
    "Analyse_Kategori",
    "Kommune",
    "Region",
    "Ukedag",
    "Årstid",
    "Helligdager",
    "HelligdagNavn",
    *measure_columns,
]
front_columns = [column for column in front_columns if column in full_calendar_panel.columns]
full_calendar_panel = full_calendar_panel[
    front_columns
    + [column for column in full_calendar_panel.columns if column not in front_columns]
]

full_calendar_panel.to_parquet(FULL_CALENDAR_PATH, index=False)

print(f"Full calendar rows: {len(full_calendar_panel):,}")
print(f"Expected rows: {expected_rows:,}")
print(f"Saved: {project_relative(FULL_CALENDAR_PATH)}")

## Closure and open-day logic

The closure rule marks a store-day as closed when total store-day sales are zero, or when suspect microsales occur on Sundays or holidays below the legacy threshold.


In [ ]:
CLOSURE_MICROSALES_THRESHOLD = 15

closure_store_day = (
    full_calendar_panel.groupby(["DatoSolgt", "AvdelingID"], as_index=False)
    .agg(
        Avdeling_total=("Antall_total", "sum"),
        Ukedag=("Ukedag", "first"),
        Helligdager=("Helligdager", "max"),
    )
)

is_sunday = closure_store_day["Ukedag"].astype(str).eq("Sunday")
is_holiday = closure_store_day["Helligdager"].eq(1)
closure_store_day["Suspect_microsales"] = (
    (is_sunday | is_holiday)
    & closure_store_day["Avdeling_total"].gt(0)
    & closure_store_day["Avdeling_total"].lt(CLOSURE_MICROSALES_THRESHOLD)
)
closure_store_day["Closed"] = (
    closure_store_day["Avdeling_total"].eq(0)
    | closure_store_day["Suspect_microsales"]
)

calendar_with_closure = full_calendar_panel.merge(
    closure_store_day[
        ["DatoSolgt", "AvdelingID", "Avdeling_total", "Closed", "Suspect_microsales"]
    ],
    on=["DatoSolgt", "AvdelingID"],
    how="left",
)

sales_columns = [
    "Antall_total",
    "Antall_campaign",
    "Antall_ordinary",
    "CampaignShare",
    "CampaignDay",
    "log_Salg",
]
calendar_with_closure.loc[calendar_with_closure["Closed"], sales_columns] = 0

calendar_with_closure["Closed"] = calendar_with_closure["Closed"].astype("int8")
calendar_with_closure["CampaignDay"] = calendar_with_closure["CampaignDay"].astype("int8")

master_without_weather = calendar_with_closure.drop(columns=["Suspect_microsales"])

closure_summary = (
    closure_store_day["Closed"]
    .value_counts(dropna=False)
    .rename_axis("Closed")
    .reset_index(name="store_days")
)
microsales_count = int(closure_store_day["Suspect_microsales"].sum())

print(f"Closed store-days: {int(closure_store_day['Closed'].sum()):,}")
print(f"Suspect microsales store-days: {microsales_count:,}")
display(closure_summary)

## Realised-weather merge

This step joins observed historical weather to the sales panel. These realised values are appropriate for regression calibration and diagnostics, but not for future operational ML forecast-weather inputs.


In [ ]:
weather_observed = pd.read_parquet(WEATHER_OBS_PATH)
required_weather_columns = [
    "Dato",
    "AvdelingID",
    "precip_val",
    "temp_mean",
    "humidity_mean",
    "wind_mean",
    "cloud_mean",
]
require_columns(weather_observed, required_weather_columns, "weather_observed")

master_without_weather["DatoSolgt"] = pd.to_datetime(master_without_weather["DatoSolgt"]).dt.normalize()
weather_observed["Dato"] = pd.to_datetime(weather_observed["Dato"]).dt.normalize()
master_without_weather["AvdelingID"] = pd.to_numeric(
    master_without_weather["AvdelingID"], errors="raise"
).astype("Int64")
weather_observed["AvdelingID"] = pd.to_numeric(weather_observed["AvdelingID"], errors="raise").astype("Int64")

weather_columns = ["precip_val", "temp_mean", "humidity_mean", "wind_mean", "cloud_mean"]
weather_merge = (
    weather_observed[["Dato", "AvdelingID", *weather_columns]]
    .drop_duplicates(subset=["Dato", "AvdelingID"])
    .rename(columns={"Dato": "DatoSolgt"})
)

master_panel = master_without_weather.merge(
    weather_merge,
    on=["DatoSolgt", "AvdelingID"],
    how="left",
)

missing_weather_rates = master_panel[weather_columns].isna().mean().sort_values(ascending=False)
duplicate_master_keys = master_panel.duplicated(["DatoSolgt", "AvdelingID", "Analyse_Kategori"]).sum()
assert duplicate_master_keys == 0, f"Duplicate master-panel keys: {duplicate_master_keys}"

master_panel.to_parquet(MASTER_PANEL_PATH, index=False)

master_panel.to_parquet(REGRESSION_PANEL_PATH, index=False)

print(f"Master panel rows: {len(master_panel):,}")
print(f"Saved: {project_relative(MASTER_PANEL_PATH)}")
print(f"Saved: {project_relative(REGRESSION_PANEL_PATH)}")
missing_weather_table = missing_weather_rates.rename("missing_rate").reset_index()
missing_weather_table.columns = ["weather_column", "missing_rate"]
display(missing_weather_table)

## Window-aware realised-weather panel

This additive long-format output repeats each store-category-day row by realised-weather aggregation window. It preserves the existing daily master and regression panels while adding the MET Nordic Analysis window proxy from notebook 01a for statistical window comparison. Forecast weather is not used here.


In [ ]:
REALISED_WINDOW_ORDER = ["full_day_00_23", "trade_08_18", "trade_08_19"]
REALISED_WINDOW_COLUMNS = [
    "temp_obs_mean",
    "temp_obs_min",
    "temp_obs_max",
    "precip_obs_sum",
    "wind_obs_mean",
    "wind_obs_max",
    "humid_obs_mean",
    "cloud_obs_mean",
    "hours_in_window",
    "expected_hours_in_window",
    "coverage_share",
    "is_partial_window",
    "is_full_window",
    "first_hour_local",
    "last_hour_local",
    "first_hour_utc",
    "last_hour_utc",
]

realised_weather_windows = pd.read_parquet(REALISED_WEATHER_WINDOWS_PATH)
required_window_columns = ["date", "AvdelingID", "aggregation_window", *REALISED_WINDOW_COLUMNS]
require_columns(realised_weather_windows, required_window_columns, "realised_weather_windows")

realised_weather_windows["date"] = pd.to_datetime(realised_weather_windows["date"]).dt.normalize()
realised_weather_windows["AvdelingID"] = pd.to_numeric(
    realised_weather_windows["AvdelingID"], errors="raise"
).astype("int64")
realised_weather_windows["aggregation_window"] = realised_weather_windows["aggregation_window"].astype(str)

unexpected_windows = sorted(
    set(realised_weather_windows["aggregation_window"].dropna()) - set(REALISED_WINDOW_ORDER)
)
if unexpected_windows:
    raise ValueError(
        f"Unexpected aggregation_window values in realised weather windows: {unexpected_windows}"
    )

duplicate_window_keys = int(
    realised_weather_windows.duplicated(["date", "AvdelingID", "aggregation_window"]).sum()
)
assert duplicate_window_keys == 0, f"Duplicate realised weather window keys: {duplicate_window_keys}"

weather_window_merge = realised_weather_windows.rename(columns={"date": "DatoSolgt"})
master_panel_windows_base = master_panel.copy()
master_panel_windows_base["DatoSolgt"] = pd.to_datetime(master_panel_windows_base["DatoSolgt"]).dt.normalize()
master_panel_windows_base["AvdelingID"] = pd.to_numeric(
    master_panel_windows_base["AvdelingID"], errors="raise"
).astype("int64")

master_panel_windows = master_panel_windows_base.merge(
    weather_window_merge,
    on=["DatoSolgt", "AvdelingID"],
    how="left",
)

window_key_columns = ["DatoSolgt", "AvdelingID", "Analyse_Kategori", "aggregation_window"]
duplicate_master_window_keys = int(master_panel_windows.duplicated(window_key_columns).sum())
assert duplicate_master_window_keys == 0, (
    f"Duplicate window-aware master-panel keys: {duplicate_master_window_keys}"
)

expected_window_set = set(REALISED_WINDOW_ORDER)
actual_window_set = set(master_panel_windows["aggregation_window"].dropna().unique())
if actual_window_set != expected_window_set:
    raise ValueError(
        f"Window-aware master panel has windows {sorted(actual_window_set)}, "
        f"expected {REALISED_WINDOW_ORDER}"
    )

expected_window_rows = len(master_panel) * len(REALISED_WINDOW_ORDER)
if len(master_panel_windows) != expected_window_rows:
    raise ValueError(
        f"Unexpected window-panel row count {len(master_panel_windows):,}; "
        f"expected {expected_window_rows:,} = master rows x {len(REALISED_WINDOW_ORDER)} windows."
    )

window_weather_missing = master_panel_windows[
    ["temp_obs_mean", "precip_obs_sum", "wind_obs_mean", "humid_obs_mean", "cloud_obs_mean"]
].isna().mean()
if float(window_weather_missing.max()) > 0:
    print(
        "Warning: missing realised window weather after merge. "
        "Inspect table below before using this output."
    )

window_panel_checks = pd.DataFrame(
    [
        {
            "check": "window_panel_rows",
            "value": len(master_panel_windows),
            "note": "Rows in long-format window-aware master panel",
        },
        {
            "check": "expected_window_panel_rows",
            "value": expected_window_rows,
            "note": "Existing master rows times three aggregation windows",
        },
        {
            "check": "n_stores",
            "value": int(master_panel_windows["AvdelingID"].nunique()),
            "note": "Unique stores",
        },
        {
            "check": "n_windows",
            "value": int(master_panel_windows["aggregation_window"].nunique()),
            "note": "Unique aggregation windows",
        },
        {
            "check": "start_date",
            "value": str(master_panel_windows["DatoSolgt"].min().date()),
            "note": "Minimum DatoSolgt",
        },
        {
            "check": "end_date",
            "value": str(master_panel_windows["DatoSolgt"].max().date()),
            "note": "Maximum DatoSolgt",
        },
        {
            "check": "duplicate_window_keys",
            "value": duplicate_master_window_keys,
            "note": "Duplicate DatoSolgt + AvdelingID + Analyse_Kategori + aggregation_window keys",
        },
        *[
            {
                "check": f"missing_rate_{column}",
                "value": float(window_weather_missing[column]),
                "note": f"Missing rate after realised-window merge for {column}",
            }
            for column in window_weather_missing.index
        ],
    ]
)

window_counts = (
    master_panel_windows.groupby("aggregation_window", observed=True)
    .agg(rows=("AvdelingID", "size"), stores=("AvdelingID", "nunique"))
    .reindex(REALISED_WINDOW_ORDER)
    .reset_index()
)

master_panel_windows.to_parquet(MASTER_PANEL_WINDOWS_PATH, index=False)
window_panel_checks.to_csv(WINDOW_PANEL_CHECKS_PATH, index=False)

print(f"Window-aware master panel rows: {len(master_panel_windows):,}")
print(f"Saved: {project_relative(MASTER_PANEL_WINDOWS_PATH)}")
print(f"Saved checks: {project_relative(WINDOW_PANEL_CHECKS_PATH)}")
display(window_counts)
display(
    window_weather_missing.rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "weather_column"})
)
display(window_panel_checks)


## Descriptive reporting panel

This reporting-only panel restores compatibility with descriptive tables that require `NettoKr`, `AntallArtikler`, and `AntallSalg`. It is built from the aggregated sales table and realised-weather master panel, and remains separate from the core regression and ML panels.


In [ ]:
descriptive_sales_source_columns = [
    "DatoSolgt",
    "AvdelingID",
    "Analyse_Kategori",
    "NettoKr",
    "AntallArtikler",
    "AntallSalg",
]
require_columns(df_final_grouped, descriptive_sales_source_columns, "df_final_grouped")

# Copy only the aggregated intermediate table, not the raw transaction data.
descriptive_sales_source = df_final_grouped.loc[
    df_final_grouped["AntallArtikler"] >= 0,
    descriptive_sales_source_columns,
].copy()
descriptive_sales_source["DatoSolgt"] = pd.to_datetime(descriptive_sales_source["DatoSolgt"]).dt.normalize()
descriptive_sales_source["AvdelingID"] = pd.to_numeric(
    descriptive_sales_source["AvdelingID"], errors="raise"
).astype("Int64")

report_memory(descriptive_sales_source, "descriptive_sales_source from grouped data")

descriptive_sales = (
    descriptive_sales_source.groupby(key_columns, observed=True, as_index=False)
    .agg(
        NettoKr=("NettoKr", "sum"),
        AntallArtikler=("AntallArtikler", "sum"),
        AntallSalg=("AntallSalg", "sum"),
    )
)

del descriptive_sales_source
gc.collect()

descriptive_panel_columns = [
    "DatoSolgt",
    "AvdelingID",
    "AvdelingTekst",
    "Region",
    "Analyse_Kategori",
    "Ukedag",
    "?rstid",
    "Helligdager",
    "HelligdagNavn",
    "Antall_total",
    "Avdeling_total",
    "Closed",
    *weather_columns,
]
descriptive_panel_columns = [column for column in descriptive_panel_columns if column in master_panel.columns]
require_columns(
    master_panel,
    [
        "DatoSolgt",
        "AvdelingID",
        "AvdelingTekst",
        "Region",
        "Analyse_Kategori",
        "Antall_total",
        "Avdeling_total",
        "Closed",
        "temp_mean",
        "precip_val",
    ],
    "master_panel",
)

descriptive_panel = master_panel[descriptive_panel_columns].merge(
    descriptive_sales,
    on=key_columns,
    how="left",
)

for column in ["NettoKr", "AntallArtikler", "AntallSalg"]:
    descriptive_panel[column] = pd.to_numeric(descriptive_panel[column], errors="coerce").fillna(0)

descriptive_closed_mask = descriptive_panel["Closed"].astype(bool)
descriptive_panel.loc[descriptive_closed_mask, ["NettoKr", "AntallArtikler", "AntallSalg"]] = 0

descriptive_panel["AntallSalg"] = pd.to_numeric(
    descriptive_panel["AntallSalg"], errors="coerce", downcast="integer"
)
for column in ["NettoKr", "AntallArtikler", "Antall_total", "Avdeling_total"]:
    if column in descriptive_panel.columns:
        descriptive_panel[column] = pd.to_numeric(
            descriptive_panel[column],
            errors="coerce",
            downcast="float",
        )

quantity_diff = (descriptive_panel["AntallArtikler"] - descriptive_panel["Antall_total"]).abs().max()
if pd.notna(quantity_diff) and quantity_diff > 1e-5:
    print(
        "Warning: AntallArtikler and Antall_total differ after reporting-panel construction; "
        f"maximum absolute difference is {quantity_diff:.6f}."
    )

descriptive_panel.to_parquet(DESCRIPTIVE_PANEL_PATH, index=False)
report_memory(descriptive_panel, "descriptive_sales_weather_panel")

print(f"Descriptive reporting panel rows: {len(descriptive_panel):,}")
print(f"Saved: {project_relative(DESCRIPTIVE_PANEL_PATH)}")


## Observed weather daily table

This output provides realised historical weather using forecast-notation date names (`target_date`, `*_obs`) for later calibration and diagnostics in the synthetic-weather notebook.


In [ ]:
weather_observed_daily_columns = [
    "DatoSolgt",
    "AvdelingID",
    "AvdelingTekst",
    "Region",
    "precip_val",
    "temp_mean",
    "humidity_mean",
    "wind_mean",
    "cloud_mean",
]
require_columns(master_panel, weather_observed_daily_columns, "master_panel")

weather_observed_daily = (
    master_panel[weather_observed_daily_columns]
    .drop_duplicates(subset=["DatoSolgt", "AvdelingID"])
    .sort_values(["AvdelingID", "DatoSolgt"])
    .reset_index(drop=True)
    .rename(
        columns={
            "DatoSolgt": "target_date",
            "precip_val": "precip_obs",
            "temp_mean": "temp_obs",
            "humidity_mean": "humidity_obs",
            "wind_mean": "wind_obs",
            "cloud_mean": "cloud_obs",
        }
    )
)

weather_consistency = weather_observed_daily.groupby(["target_date", "AvdelingID"]).nunique(dropna=False)
max_unique_per_store_day = int(weather_consistency.max().max()) if not weather_consistency.empty else 0
assert max_unique_per_store_day <= 1, "Observed weather daily table has inconsistent store-day values."

weather_observed_daily.to_parquet(WEATHER_OBSERVED_DAILY_PATH, index=False)

print(f"Observed weather daily rows: {len(weather_observed_daily):,}")
print(f"Saved: {project_relative(WEATHER_OBSERVED_DAILY_PATH)}")

## Master-panel checks

The final checks are saved as a small traceability CSV. They summarize construction integrity but do not replace later analysis-specific sample checks.


In [ ]:
check_rows = []


def add_check(name, value, note=""):
    check_rows.append({"check": name, "value": value, "note": note})


add_check(
    "transaction_rows_raw",
    raw_transaction_rows,
    "Raw rows processed from selected Parquet columns",
)
add_check(
    "transaction_rows_after_filtering",
    filtered_transaction_rows,
    "Rows after exclusions and before aggregation",
)
add_check(
    "df_daily_rows",
    df_daily_rows,
    "Item/category/store/day aggregation rows; not kept in memory after final grouping",
)
add_check("df_final_grouped_rows", df_final_grouped_rows, "Grouped category/campaign rows")
add_check(
    "df_catday_rows",
    len(df_catday),
    "Observed store-category-day rows before calendar expansion",
)
add_check("full_calendar_rows", len(full_calendar_panel), "Complete date-store-category grid rows")
add_check(
    "master_panel_rows",
    len(master_panel),
    "Rows after closure logic and realised weather merge",
)
add_check(
    "descriptive_panel_rows",
    len(descriptive_panel),
    "Rows in reporting-only descriptive sales/weather panel",
)
add_check("master_panel_start_date", str(master_panel["DatoSolgt"].min().date()), "Minimum DatoSolgt")
add_check("master_panel_end_date", str(master_panel["DatoSolgt"].max().date()), "Maximum DatoSolgt")
add_check("n_stores", master_panel["AvdelingID"].nunique(), "Unique stores")
add_check("n_categories", master_panel["Analyse_Kategori"].nunique(), "Unique analysis categories")
add_check(
    "duplicate_master_keys",
    int(duplicate_master_keys),
    "Duplicate DatoSolgt + AvdelingID + Analyse_Kategori keys",
)
add_check("closed_store_days", int(closure_store_day["Closed"].sum()), "Store-days marked closed")
add_check("suspect_microsales_store_days", microsales_count, "Temporary closure diagnostic count")

for column in [
    "DatoSolgt",
    "AvdelingID",
    "AvdelingTekst",
    "Region",
    "Analyse_Kategori",
    "Antall_total",
    "Closed",
]:
    add_check(f"missing_{column}", int(master_panel[column].isna().sum()), f"Missing values in {column}")

for column in ["NettoKr", "AntallArtikler", "AntallSalg"]:
    add_check(
        f"missing_descriptive_{column}",
        int(descriptive_panel[column].isna().sum()),
        f"Missing values in descriptive panel column {column}",
    )

for column in weather_columns:
    add_check(f"missing_rate_{column}", float(missing_weather_rates[column]), f"Missing rate for {column}")

checks_df = pd.DataFrame(check_rows)
checks_df.to_csv(CHECKS_PATH, index=False)

print(f"Saved checks: {project_relative(CHECKS_PATH)}")
display(checks_df)


## Downstream use

After this notebook has been run successfully, use `data/processed/master_panel_realized_weather.parquet` for realised-weather analysis, `data/processed/master_panel_realized_weather_windows.parquet` for weather-window comparison, `data/processed/regression_panel_realized_weather.parquet` for category-level realised-weather regressions, `data/processed/descriptive_sales_weather_panel.parquet` for descriptive reporting fields, and `data/processed/weather_observed_daily.parquet` for the synthetic-weather notebook. Continue with `notebooks/02_basket_analysis.ipynb`. Realised target-day weather from this notebook is not a valid operational ML forecast-weather input; later ML notebooks must use forecast-weather features indexed by `origin_date`, `target_date`, and `horizon`.
